In [1]:
!pip install xlrd

In [2]:
import pandas as pd
df = pd.read_excel("GSAF5.xls")

In [3]:
df.columns

Index(['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity',
       'Name', 'Sex', 'Age', 'Injury', 'Fatal Y/N', 'Time', 'Species ',
       'Source', 'pdf', 'href formula', 'href', 'Case Number', 'Case Number.1',
       'original order', 'Unnamed: 21', 'Unnamed: 22'],
      dtype='object')

# Cleaning 

In [4]:
# Modifying column names
df = df.drop(['Unnamed: 21', 'Unnamed: 22'], axis=1)

# Clean column names
df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('/', '', regex=False))

In [5]:
# viewing data
df.isnull().sum().sort_values()

date                 0
year                 2
type                18
source              20
injury              35
country             50
name               218
pdf                283
original_order     283
case_number        284
case_number.1      285
href               286
href_formula       288
state              487
fatal_yn           561
location           567
sex                578
activity           583
age               2994
species           3131
time              3527
dtype: int64

In [6]:
df.head(3)

,date,year,type,country,state,location,activity,name,sex,age,...,fatal_yn,time,species,source,pdf,href_formula,href,case_number,case_number.1,original_order
0,18th March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,Unknown,M,39,...,N,1715hrs,Great White Shark,US Sun: Mendocino Coast News:Melissa Michaelson:,NaN,NaN,NaN,NaN,NaN,NaN
1,14th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Swimming,Unknown,F,?,...,N,1015hrs,Unknown shark 5ft (1.5m),People Magazine: Kevin McMurray Trackingsharks...,NaN,NaN,NaN,NaN,NaN,NaN
2,10th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Wing Foiling,Dave Daniell,M,?,...,N,0800hrs,Great White Shark 10ft (3m),Perth Now: Kevin McMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN


# Name

In [7]:
# Replace Known values
df['name'] = df['name'].replace(['Anonymous', 'Unidentified', 'Not stated','a sailor', 'a pearl diver', 'a native', 'a soldier','a native fisherman' ,'native', 'males', 'a school teacher', 'soldier', 'unknown', 'a local dignitary', 'Japanese diver', 'teen', 'a pilot', '"a youth"', 'diver', 'surfer', 'fisherman', 'fishermen', 'swimmer', 'bather', 'bathers', 'skindiver', '2 fishermen', '2 divers'], 'Unknown')

In [8]:
# Relaces any other values
df.loc[df['name'].str.contains(r'^\d+|\b(?:boy|girl|male|female|man|woman|men|women|child|diver|surfer|boat)\b', case=False,na=False),'name'] = 'Unknown'

In [9]:
df['name'].value_counts().head(5)

name
Unknown          1499
sailor             10
soldier             3
M.C.                3
Andre Hartman       3
Name: count, dtype: int64

In [10]:
df['name'].isnull().sum()

np.int64(218)

In [11]:
# Filling in null values with 'Unknown'
df['name'] = df['name'].fillna('Unknown')

In [12]:
df['name'].value_counts().head(10)

name
Unknown          1717
sailor             10
soldier             3
M.C.                3
Andre Hartman       3
crew                3
John Williams       3
dinghy              3
Gary Johnson        3
Erika Lane          2
Name: count, dtype: int64

# Sex

In [13]:
# standardize
df['sex'] = df['sex'].str.upper().str.strip()
# clean values
df['sex'] = df['sex'].replace({'Male': 'M','Female': 'F','LLI': 'M' ,'N': None,'?': None,'.': None})
# fill missing
df['sex'] = df['sex'].fillna('Unknown')

df.head(2)

,date,year,type,country,state,location,activity,name,sex,age,...,fatal_yn,time,species,source,pdf,href_formula,href,case_number,case_number.1,original_order
0,18th March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,Unknown,M,39,...,N,1715hrs,Great White Shark,US Sun: Mendocino Coast News:Melissa Michaelson:,NaN,NaN,NaN,NaN,NaN,NaN
1,14th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Swimming,Unknown,F,?,...,N,1015hrs,Unknown shark 5ft (1.5m),People Magazine: Kevin McMurray Trackingsharks...,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.sex.unique()

array(['M', 'F', 'Unknown', 'M X 2'], dtype=object)

In [15]:
df[df['sex']=='M X 2']

,date,year,type,country,state,location,activity,name,sex,age,...,fatal_yn,time,species,source,pdf,href_formula,href,case_number,case_number.1,original_order
3716,23-Oct-1962,1982.0,Sea Disaster,USA,Carolina coast,NaN,Yacht Trashman capsized in storm,John Lippoth & Mark Adams,M X 2,NaN,...,Y x 2,NaN,NaN,"New York Times, 10/31/1982",1982.10.00-Trashman.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1982.10.23,1982.10.23,3362.0


In [16]:
# Changing the value from 'M X 2' to 2 rows
row = df.loc[3716].copy()

names = [i.strip() for i in row['name'].split('&')]

new_rows = []

for name in names:
    new_row = row.copy()
    new_row['name'] = name
    new_row['sex'] = 'M'   
    new_rows.append(new_row)

# Removeing original row
df = df.drop(index=3716)

# Add new rows
df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

In [17]:
df.sex.unique()

array(['M', 'F', 'Unknown'], dtype=object)

# Age

In [18]:
# Replace the's' 
df['age'] = df['age'].astype(str).str.replace('s', '', regex=False)

# convert to num
df['age'] = pd.to_numeric(df['age'], errors='coerce')

In [19]:
df['age'] = df['age'].fillna(df['age'].median())
df['age'] = df['age'].astype(int)

In [20]:
df.age.unique()

array([39, 24, 55, 13, 11, 27, 12, 26, 56, 25, 61, 40, 14, 54, 48, 57,  8,
       63,  9, 19,  7, 85, 69, 18, 66, 21, 37, 16, 20, 42, 45, 30, 29, 35,
       58, 17, 36, 23, 28, 38, 68, 33, 15, 41, 43, 49, 46, 65, 64, 32, 10,
       62, 22, 52, 44, 47, 59, 50, 34, 77, 60, 73, 67,  6, 53, 51, 31, 71,
       75, 70,  4, 74,  3, 82, 72,  5, 86, 84, 87,  1, 81, 78])

In [21]:
df['age'].isnull().sum()

np.int64(0)

# Injury

In [22]:
df['injury'].isnull().sum()

np.int64(35)

In [23]:
df.injury.unique()

array(['Injuries to both legs and feet', 'Minor injuries',
       'None reported damage to board', ...,
       'FATAL, leg stripped of flesh  ',
       'FATAL, knocked overboard by tail of shark & carried off by shark ',
       'FATAL. "Shark bit him in half, carrying away the lower extremities" '],
      shape=(4194,), dtype=object)

In [24]:
# Standardize text in column
df['injury'] = df['injury'].str.strip().str.lower()

# Create function to change values
def clean_column(x):
    if pd.isna(x):
        return 'Unknown'
    x = x.lower()
    if 'fatal' in x:
        return 'Fatal'
    elif 'no injury' in x or 'no injuries' in x:
        return 'No injury'
    elif 'minor' in x:
        return 'Minor'
    elif any(word in x for word in ['stripped', 'bitten', 'extremities', 'extremities']):
        return 'Severe'
    else:
        return 'Unknown'
df['injury'] = df['injury'].apply(clean_column)   

In [25]:
df.injury.unique()

array(['Unknown', 'Minor', 'Severe', 'No injury', 'Fatal'], dtype=object)

In [26]:
df['injury'].isnull().sum()

np.int64(0)

# Fatal

In [27]:
df['fatal_yn'].isnull().sum()

np.int64(561)

In [28]:
df[df['fatal_yn'].isna()]

,date,year,type,country,state,location,activity,name,sex,age,...,fatal_yn,time,species,source,pdf,href_formula,href,case_number,case_number.1,original_order
175,25 Aug 2023,2023.0,Unprovoked,AUSTRALIA,New South Wales,"Lighthouse Beach, Port Macquarie",Surfing,Toby Begg,M,44,...,NaN,10h00,"White shark, 3.8-4.2m","B. Myatt, & M. Michaelson, GSAF",NaN,NaN,NaN,NaN,NaN,NaN
178,21 Aug-2023,2023.0,Questionable,BAHAMAS,New Providence Isoad,"Saunders Beach, Nassau",NaN,Unknown,M,24,...,NaN,Morning,NaN,"The Tribune, 8/21/2023",NaN,NaN,NaN,NaN,NaN,NaN
203,07-Jun-2023,2023.0,Unprovoked,BAHAMAS,Freeport,Shark Junction,Scuba diving,Heidi Ernst,F,73,...,NaN,13h00,Caribbean rreef shark,"J. Marchand, GSAF",NaN,NaN,NaN,NaN,NaN,NaN
235,02-Mar-2023,2023.0,Unprovoked,SEYCHELLES,Praslin Island,NaN,Snorkeling,Arthur …,M,6,...,NaN,Afternoon,Lemon shark,"Midlibre, 3/18/2023",NaN,NaN,NaN,NaN,NaN,NaN
239,18-Feb-2023,2023.0,Questionable,ARGENTINA,Patagonia,Chubut Province,NaN,Diego Barría,M,32,...,NaN,NaN,NaN,"El Pais, 2/27/2023",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6928,1733,1733.0,Invalid,ICELAND,Bardestrand,Talkknefiord,NaN,Unknown,Unknown,24,...,NaN,NaN,Shark involvement prior to death unconfirmed,E. Olafsen,1733.00.00-Iceland.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1733.00.00,1733.00.00,152.0
6929,1723,1723.0,Unprovoked,ROATAN,NaN,NaN,NaN,Philip Ashton,M,24,...,NaN,NaN,NaN,"C.Moore, GSAF",1730.00.00-Ashton.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1723.00.00,1723.00.00,151.0
6935,Late 1600s Reported 1728,1642.0,Invalid,GUINEA,NaN,NaN,Went overboard,crew member of the Nieuwstadt,M,24,...,NaN,NaN,Questionable,"History of the Pyrates, by D. Defoe, Vol. 2, p.28",1642.00.00.b-Nieuwstadt.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1642.00.00.b,1642.00.00.b,145.0
6960,Before 1824,0.0,Unprovoked,AUSTRALIA,Queensland,Newstead,Swimming,Eullah,F,24,...,NaN,NaN,NaN,"B. Myatt, GSAF",ND-0155-2AboriginalChildren.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND-0155,ND-0155,122.0


In [29]:
df.fatal_yn.unique()

array(['N', 'Y', 'F', 'M', nan, 'n', 'Nq', 'UNKNOWN', 2017, ' N', 'N ',
       'y', 'Y x 2'], dtype=object)

In [30]:
df['fatal_yn'] = df['fatal_yn'].replace({'n':'N', ' N': 'N','N ':'N', 'Y x 2': 'Y', 'F':'Unknown','y':'Y' ,'M': 'Unknown', 'UNKNOWN':'Unknown','Nq':'Unknown'})

In [31]:
df[df['fatal_yn']==2017]

,date,year,type,country,state,location,activity,name,sex,age,...,fatal_yn,time,species,source,pdf,href_formula,href,case_number,case_number.1,original_order
1536,10-Jun-2012,2012.0,Provoked,ITALY,Sardinia,Muravera,Attempting to rescue an injured & beached shark,Giorgio Zara,M,57,...,2017,Morning,"Blue shark, 2.5m","D. Puddo, 6/11/2012",2012.06.10-Zara.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,2012.06.10,2012.06.10,5542.0


In [32]:
df.loc[df['fatal_yn'] == 2017, ['fatal_yn', 'injury', 'name', 'activity']]

,fatal_yn,injury,name,activity
1536,2017,Unknown,Giorgio Zara,Attempting to rescue an injured & beached shark


In [33]:
# Changing to No
df.loc[df['fatal_yn'] == 2017, 'fatal_yn'] = 'N'

In [34]:
df.loc[df['fatal_yn'].isna() & (df['injury'] == 'Fatal'), 'fatal_yn'] = 'Y'

In [35]:
df.loc[df['fatal_yn'].isna() & (df['injury'] == 'No injury'), 'fatal_yn'] = 'N'

In [36]:
df['fatal_yn'] = df['fatal_yn'].fillna('Unknown')

In [37]:
df.fatal_yn.unique()

array(['N', 'Y', 'Unknown'], dtype=object)

# Time

In [38]:
df.tail(5)

,date,year,type,country,state,location,activity,name,sex,age,...,fatal_yn,time,species,source,pdf,href_formula,href,case_number,case_number.1,original_order
7078,1900-1905,0.0,Unprovoked,USA,North Carolina,Ocracoke Inlet,Swimming,Coast Guard personnel,M,24,...,Y,NaN,NaN,"F. Schwartz, p.23; C. Creswell, GSAF",ND-0003-Ocracoke_1900-1905.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0003,ND.0003,4.0
7079,1883-1889,0.0,Unprovoked,PANAMA,NaN,"Panama Bay 8ºN, 79ºW",NaN,Jules Patterson,M,24,...,Y,NaN,NaN,"The Sun, 10/20/1938",ND-0002-JulesPatterson.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0002,ND.0002,3.0
7080,1845-1853,0.0,Unprovoked,CEYLON (SRI LANKA),Eastern Province,"Below the English fort, Trincomalee",Swimming,Unknown,M,15,...,Y,NaN,NaN,S.W. Baker,ND-0001-Ceylon.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,ND.0001,ND.0001,2.0
7081,23-Oct-1962,1982.0,Sea Disaster,USA,Carolina coast,NaN,Yacht Trashman capsized in storm,John Lippoth,M,24,...,Y,NaN,NaN,"New York Times, 10/31/1982",1982.10.00-Trashman.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1982.10.23,1982.10.23,3362.0
7082,23-Oct-1962,1982.0,Sea Disaster,USA,Carolina coast,NaN,Yacht Trashman capsized in storm,Mark Adams,M,24,...,Y,NaN,NaN,"New York Times, 10/31/1982",1982.10.00-Trashman.pdf,http://sharkattackfile.net/spreadsheets/pdf_di...,http://sharkattackfile.net/spreadsheets/pdf_di...,1982.10.23,1982.10.23,3362.0


# Species

In [39]:
df.species.unique()

array(['Great White Shark', 'Unknown shark 5ft (1.5m)',
       'Great White Shark 10ft (3m)', ..., "12' tiger shark",
       'Blue pointers',
       'Said to involve a grey nurse shark that leapt out of the water and  seized the boy but species identification is questionable'],
      shape=(1745,), dtype=object)

In [40]:
df['species'] = df['species'].str.lower().str.strip()

In [41]:
# Function for species --- keeping skark seperate to unknown 
def species(x):
    if pd.isna(x):
        return 'unknown'
    x = x.lower()  
    if 'white' in x:
        return 'white shark'
    elif 'tiger' in x:
        return 'tiger shark'
    elif 'bull' in x:
        return 'bull shark'
    elif 'shark' in x:
        return 'other shark'
    else:
        return 'unknown'

df['species'] = df['species'].apply(species)

In [42]:
df.species.unique()

array(['white shark', 'other shark', 'unknown', 'tiger shark',
       'bull shark'], dtype=object)

In [43]:
df['species'].isnull().sum()

np.int64(0)

In [44]:
df.head(15)

,date,year,type,country,state,location,activity,name,sex,age,...,fatal_yn,time,species,source,pdf,href_formula,href,case_number,case_number.1,original_order
0,18th March,2026.0,Unprovoked,USA,California,Big River Beach Mendocino County,Surfing,Unknown,M,39,...,N,1715hrs,white shark,US Sun: Mendocino Coast News:Melissa Michaelson:,NaN,NaN,NaN,NaN,NaN,NaN
1,14th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Swimming,Unknown,F,24,...,N,1015hrs,other shark,People Magazine: Kevin McMurray Trackingsharks...,NaN,NaN,NaN,NaN,NaN,NaN
2,10th March,2026.0,Unprovoked,Australia,Western Australia,Exmouth,Wing Foiling,Dave Daniell,M,24,...,N,0800hrs,white shark,Perth Now: Kevin McMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN
3,5th March,2026.0,Unprovoked,Australia,Queensland,Lady Elliott Island,Snorkeling,Unknown,M,24,...,N,0815hrs,unknown,News.com.au: ABC News: Andy Currie,NaN,NaN,NaN,NaN,NaN,NaN
4,22nd February,2026.0,Unprovoked,New Caledonia,Noumea,Anse Vata near Point Magnin,Wing Foiling,Cyril Chevalier,M,55,...,Y,?,tiger shark,Johannes Marchand: Andy Currie,NaN,NaN,NaN,NaN,NaN,NaN
5,6th February,2026.0,Unprovoked,Cayman Islands,Little Cayman Island,Caribbean Sea,Diving,Bekley White,M,24,...,N,?,tiger shark,Kevin McMurray Trackingshark.som,NaN,NaN,NaN,NaN,NaN,NaN
6,29th January,2026.0,Unprovoked,Brazil,Recife,Del Chifre Beach in Olinda,Swimming,Deivson Rocha Dantas,M,13,...,Y,?,tiger shark,Kevin McMurray Trackingsharks.com: TV Globo: P...,NaN,NaN,NaN,NaN,NaN,NaN
7,29th January,2026.0,Unprovoked,Australia,NSW,Angels Beach East Ballina,Surfing,Unknown,M,24,...,N,1100hrs,unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN
8,24th January,2026.0,Unprovoked,Australia,Tasmania,Cooee Beach west of Burnie,Swimming,Megan Stokes,F,24,...,N,1815hrs,other shark,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN
9,20th January,2026.0,Unprovoked,Australia,NSW,Point Plomber North of Port Macquarie,Surfing,Paul Zvirdinas,M,39,...,N,0830hrs,bull shark,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN
